# Preprocessing

## Análisis y preprocesado de los datos

Iniciamos leyendo los datos, que contienen varios errores a arreglar durante el preprocesado

In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from functions_scripts.load import read_data
from classes.Preprocessor import Preprocessor

pd.set_option('future.no_silent_downcasting', True)

df = read_data(filepath="../data/Autism-Adult-Data.csv")

preprocessor = Preprocessor(df=df)

#### Reemplazo de nombres erróneos de columnas
Los nombres de columnas 'jundice' y 'Austim' son incorrectos, ya que se refieren a 'jaundice' y 'family_pdd' respectivamente.

In [5]:
# Las columnas Jundice y Austim se encuentran en las posiciones 14 y 15

idxs = [14, 15]

print("ANTES:")
preprocessor.column_get_unique(idxs)

preprocessor.column_rename(idxs, ["jaundice", "family_pdd"])

print("\nDESPUES:")
preprocessor.column_get_unique(idxs)

ANTES:
Unique vals in jundice: ['no' 'yes']
Unique vals in austim: ['no' 'yes']

DESPUES:
Unique vals in jaundice: ['no' 'yes']
Unique vals in family_pdd: ['no' 'yes']


#### Error en representación de valores perdidos

Los valores perdidos en este dataset están representados como ?, y nos sirve que estén representados como NaN, ya que sino las columnas quedarán como dtype='object' y necesitamos que sean numéricas.

Demostración del error:

In [6]:
preprocessor.column_get_unique("age") # Debería ser numérica pero no lo es
preprocessor.column_get_unique("result") # Debería ser numérica (en este caso no hace falta arreglarla 
                            # porque no hay valores perdidos)

Unique vals in age: ['17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30'
 '31' '32' '33' '34' '35' '36' '37' '38' '383' '39' '40' '41' '42' '43'
 '44' '45' '46' '47' '48' '49' '50' '51' '52' '53' '54' '55' '56' '58'
 '59' '60' '61' '64' '?']
Unique vals in result: [ 0  1  2  3  4  5  6  7  8  9 10]


In [7]:
preprocessor.column_to_numeric("age")


Confirmación de que se realizó la operación correctamente:

In [8]:
preprocessor.column_get_unique("age") # Debería ser numérica, ahora lo es

Unique vals in age: [17.0 18.0 19.0 20.0 21.0 22.0 23.0 24.0 25.0 26.0 27.0 nan 27.0 28.0 29.0
 30.0 nan 30.0 31.0 32.0 33.0 34.0 35.0 36.0 37.0 38.0 39.0 40.0 41.0 42.0
 43.0 44.0 45.0 46.0 47.0 48.0 49.0 50.0 51.0 52.0 53.0 54.0 55.0 56.0
 58.0 59.0 60.0 61.0 64.0 383.0]


### Errores de nombrado y columna innecesaria

Las columnas ID y age_desc no aportan información, ya que ID contiene valores diferentes incrementales por cada uno de los datos, y age_desc contiene un único valor diferente: '18 and more'

In [9]:
preprocessor.column_get_unique("age_desc") # Debería ser numérica, ahora lo es

print(f"# of rows in dataframe: {df.shape[0]}")
print(f"# of unique vals in ID: {len(np.unique(df.id))}")

preprocessor.column_del(["id", "age_desc"])

Unique vals in age_desc: ['18 and more']
# of rows in dataframe: 704
# of unique vals in ID: 704


### Columnas listas para valores faltantes, outliers e inconsistencias

Vamos a hacer una breve muestra de como quedaron los datos

In [10]:
preprocessor.describe_data()



Columna: A1_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.721591
std        0.448535
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: A1_Score, dtype: float64


Columna: A2_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.453125
std        0.498152
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A2_Score, dtype: float64


Columna: A3_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.457386
std        0.498535
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A3_Score, dtype: float64


Columna: A4_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.495739
std        0.500337
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.00

### Categorico a numérico

Tenemos varias columnas binarias cuyos valores son {'no', 'yes'}, {'f', 'm'}, ...

Estas columnas las reemplazaremos por columnas enteras con estos reemplazamientos:

    no, f = 0
    yes, m = 1

In [11]:
idxs_categoric_binary_columns = [11, 13, 14, 16, 19]

print("ANTES:")
preprocessor.column_get_unique(idxs_categoric_binary_columns)
preprocessor.column_binary_categoric_to_numeric(idxs_categoric_binary_columns)

print("\nDESPUES:")
preprocessor.column_get_unique(idxs_categoric_binary_columns)

ANTES:
Unique vals in gender: ['f' 'm']
Unique vals in jaundice: ['no' 'yes']
Unique vals in family_pdd: ['no' 'yes']
Unique vals in used_app_before: ['no' 'yes']
Unique vals in Class/ASD: ['NO' 'YES']

DESPUES:
Unique vals in gender: [0 1]
Unique vals in jaundice: [0 1]
Unique vals in family_pdd: [0 1]
Unique vals in used_app_before: [0 1]
Unique vals in Class/ASD: [0 1]
